In [2]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
class_mapping = {'white': 0,'red': 1,'yellow': 2,'blue': 3}
def convert(size, box):
    width, height = size
    xmin, ymin, xmax, ymax = box
    dw = 1.0 / width
    dh = 1.0 / height
    x_center = (xmin + xmax) / 2.0 * dw
    y_center = (ymin + ymax) / 2.0 * dh
    box_width = (xmax - xmin) * dw
    box_height = (ymax - ymin) * dh
    return (x_center, y_center, box_width, box_height)

In [3]:
def xml_to_txt_batch(xml_folder, txt_folder):
    xml_files = [f for f in os.listdir(xml_folder) if f.endswith('.xml')]
    success_count = 0
    for xml_file in xml_files:
        xml_path = os.path.join(xml_folder, xml_file)
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            size_elem = root.find('size')
            width = int(size_elem.find('width').text)
            height = int(size_elem.find('height').text)
            size = (width, height)
            txt_filename = xml_file.replace('.xml', '.txt')
            txt_path = os.path.join(txt_folder, txt_filename)
            
            with open(txt_path, 'w', encoding='utf-8') as f:
                for obj in root.iter('object'):
                    name = obj.find('name').text
                    if name in class_mapping:
                        class_id = class_mapping[name]
                    else:
                        print(f"警告: {xml_file} 中发现未知类别 '{name}'，跳过该目标")
                        continue
                    bndbox = obj.find('bndbox')
                    xmin = int(bndbox.find('xmin').text)
                    ymin = int(bndbox.find('ymin').text)
                    xmax = int(bndbox.find('xmax').text)
                    ymax = int(bndbox.find('ymax').text)
                    box_coords = (xmin, ymin, xmax, ymax)
                    yolo_coords = convert(size, box_coords)
                    line = f"{class_id} {yolo_coords[0]:.6f} {yolo_coords[1]:.6f} {yolo_coords[2]:.6f} {yolo_coords[3]:.6f}\n"
                    f.write(line)
            print(f"'{xml_file}'文件已成功转换")
            success_count += 1
        except Exception as e:
            print(f"错误: 处理 '{xml_file}' 时发生异常: {str(e)}")
    print(f"\n转换完成！成功转换 {success_count}/{len(xml_files)} 个文件")

In [4]:
xml_folder = r"C:\Users\wbh\Desktop\标注实验图片"  # XML文件夹路径
txt_folder = r"C:\Users\wbh\Desktop\标注实验图片\txt"  # TXT输出文件夹路径
xml_to_txt_batch(xml_folder, txt_folder)

'001.xml'文件已成功转换
'002.xml'文件已成功转换
'003.xml'文件已成功转换
'004.xml'文件已成功转换
'005.xml'文件已成功转换
'006.xml'文件已成功转换
'007.xml'文件已成功转换
'008.xml'文件已成功转换
'009.xml'文件已成功转换
'010.xml'文件已成功转换
'011.xml'文件已成功转换
'012.xml'文件已成功转换
'013.xml'文件已成功转换
'014.xml'文件已成功转换
'015.xml'文件已成功转换

转换完成！成功转换 15/15 个文件
